# 13 pamoka – Agento atmintis su Cognee žinių grafais


## Nustatymas

Ši užrašų knyga parodo, kaip sukurti intelektualų **programavimo asistentą** su nuolatine atmintimi, naudojant [**Cognee**](https://www.cognee.ai/) žinių grafus ir **Microsoft Agent Framework** (MAF).

Cognee paverčia nestruktūruotą tekstą į struktūruotą, užklausomą žinių grafiką, paremta vektorinėmis įdėjomis — suteikiant jūsų agentui turtingą, santykius suprantančią ilgalaikę atmintį.

### Ko Išmoksite
1. **Kurti Žinių Grafus**: Paversti programuotojų profilius ir gerąją praktiką į struktūruotas, užklausomas žinias.
2. **Integruoti Cognee su MAF**: Naudoti `@tool` funkcijas, kad MAF agentas galėtų užklausti Cognee žinių grafą.
3. **Seansų Suvokiančios Pokalbiai**: Išlaikyti kontekstą kelių klausimų toje pačioje sesijoje metu.
4. **Ilgalaikė Atmintis**: Išsaugoti svarbias žinias sesijų metu ir jas atkurti naujuose pokalbiuose.

### Reikalavimai
- Python 3.9+
- Vietoje veikiantis Redis (`docker run -d -p 6379:6379 redis`) sesijų valdymui
- LLM API raktas (pvz., OpenAI) — nustatyti `LLM_API_KEY` faile `.env`
- `CACHING=true` faile `.env` (reikalinga Cognee sesijoms)
- Microsoft Foundry projektas su įdiegta pokalbių modeliu
- `AZURE_AI_PROJECT_ENDPOINT` ir `AZURE_AI_MODEL_DEPLOYMENT_NAME` faile `.env`
- Prisijungta prie Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Agentų atminties tipai

Šiame užrašų knygelėje nagrinėjami tie patys trys atminties tipai kaip pagrindinėje 13 pamokos užrašų knygelėje, tačiau kaip ilgalaikės atminties pagrindą naudojamas Cognee:

| Atminties tipas | Mechanizmas | Gyvavimo laikas |
|---|---|---|
| **Darbinė** | `agent.create_session()` (MAF) | Vienas pokalbis |
| **Trumpalaikė** | Cognee sesijos talpykla (Redis) | Viena sesija |
| **Ilgalaikė** | Cognee žinių grafas + vektorės | Nuolatinė |

### Cogneės atminties architektūra
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Paruoškite Cognee saugyklą


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## 1 dalis — Žinių bazės kūrimas

Mes renkamės tris duomenų tipus, kad sukurtume išsamią žinių bazę mūsų kodavimo asistentui:

1. **Kūrėjo profilis** — asmeninė patirtis ir techninis išsilavinimas
2. **Python geriausios praktikos** — Python Zen su praktiškomis gairėmis
3. **Istorinės pokalbių sesijos** — praeities klausimų ir atsakymų sesijos tarp kūrėjų ir AI asistentų


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Vizualizuokite Žinių Grafą

Cognee gali sugeneruoti interaktyvią HTML vizualizaciją apie išgautas subjektus ir ryšius.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Praturtinkite atmintį su Memify

`memify()` analizuoja žinių grafiką ir generuoja intelektualias taisykles — identifikuodama modelius, geriausias praktikas ir ryšius tarp sąvokų.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## 2 dalis — MAF agentas su Cognee įrankiais

Dabar sukursime MAF agentą, galintį užklausti Cognee žinių grafą per `@tool` funkcijas. Tai leidžia agentui pasinaudoti visomis grafu pagrįstos semantinės paieškos galimybėmis, išlaikant pokalbio kontekstą per sesijas.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Darbinė atmintis su sesijomis

`AgentSession` (sukuriama per `agent.create_session()`) suteikia darbinę atmintį sesijos metu. Agentas gali grįžti prie ankstesnių žinučių, taip pat užklausti Cognee ilgalaikio žinių grafo.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nauja sesija — ilgalaikė atmintis išlieka

Pradedant naują sesiją, darbo atmintis išvaloma, bet Cognee žinių grafas vis dar prieinamas. Agentas gali atkurti tą pačią ilgalaikę informaciją visiškai naujoje pokalbyje.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Santrauka

Šiame užrašų knygelėje sukūrėte kodavimo asistentą, kuris sujungia **MAF darbinę atmintį** (`agent.create_session()`) su **Cognee ilgalaike žinių grafu**.

### Ko Išmokote
1. **Žinių grafo konstrukcija**: Cognee naudoja nestruktūruotą tekstą ir kuria grafą + vektorinę atmintį.
2. **Grafo papildymas su memify**: Išvestiniai faktai ir turtingesni ryšiai virš jūsų esamo grafo.
3. **MAF + Cognee integracija**: `@tool` funkcijos leidžia MAF agentui natūraliai užklausti Cognee grafą.
4. **Darbinė atmintis + ilgalaikė atmintis**: `AgentSession` (per `agent.create_session()`) suteikia sesijos kontekstą, o Cognee suteikia nuolatinę informaciją.
5. **Filtruota paieška su NodeSets**: Orientavimasis į konkrečias žinių grafo dalis (pvz., tik principus).

### Pagrindinės Išvados
- **Cognee** paverčia žalią tekstą į struktūrizuotą, santykiais pagrįstą atmintį — galingesnę nei plokščias vektorinis saugykla.
- **`@tool` funkcijos** sklandžiai sujungia MAF agentus su išorinėmis žinių sistemomis.
- **`AgentSession`** (per `agent.create_session()`) atskiria konversacijos kontekstą nuo ilgalaikės žinių bazės.
- Tas pats žinių grafas aptarnauja kelias sesijas ir agentus.

### Realaus Pasaulio Pritaikymai
- **Kūrėjų pagalbininkai**: Kodo peržiūra, incidentų analizė, architektūros asistentai
- **Klientų aptarnavimo pagalbininkai**: Pagalbos agentai, naudodami produkto dokumentus, DUK ir CRM pastabas
- **Vidiniai ekspertų pagalbininkai**: Politikos, teisės ar saugumo asistentai, atliekantys analizę pagal gaires
- **Vieninga duomenų sluoksniai**: Apjungti struktūruotus ir nestruktūruotus duomenis į vieną užklausose naudojamą grafą

### Tolimesni Žingsniai
- Eksperimentuoti su laiko suvokimu Cognee
- Apibrėžti OWL ontologiją domeno specifinei grafo kokybei
- Įdiegti naudotojų grįžtamojo ryšio ciklus, kad pagerintumėte paiešką laikui bėgant
- Išplėsti iki daugiaagentinių sistemų, dalijančių tą patį Cognee atminties sluoksnį


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
